## Первая нейросетевая модель

В 2003 году Josua Benjio презентовал работу "A Neural Probabilistic Language Model" [(Benjio et al, 2003)](https://jmlr.org/papers/volume3/tmp/bengio03a.pdf) - с этого момента языковое моделирование стало нейросетевым

N-граммные модели страдали от проклятия размерности:
- Во-первых, число возможных сочетаний слов астрономическое (`|V|ⁿ`), поэтому большинство реальных n-грамм просто ни разу не встречается в обучающих данных и получает нулевую вероятность: приходилось делать сглаживание и откаты (backoff)
- Во-вторых, слова в таких моделях - атомарные дискретные символы без семантики («кот» и «собака» для них различаются ровно так же, как «кот» и «пылесос», просто разные индексы в словаре). Из-за этого модель не умеет нормально обобщать

Бенжио предложил сопоставить каждому слову словаря обучаемый вещественный вектор признаков (в их экспериментах — порядка 30–100 измерений), то есть распределённое представление. И дальше — выразить вероятность последовательности через эти векторы, причём векторы и саму вероятностную функцию учить совместно, одной нейросетью.

<img src="img/benjio.png" width=400>

Это мгновенно дало обобщение: если в ходе обучения «кот» и «собака» получают близкие векторы, то модель, видевшая «кот сидел на полу», автоматически назначит разумную вероятность фразе «собака сидела на полу», даже если её в данных не было. Похожие слова → близкие векторы → переносимое знание. Вот так непрерывное пространство признаков лечит проклятие размерности.

__Архитектура__<br>
Обычная полносвязная сеть (MLP) с фиксированным окном контекста. На схеме пример для n=3

Представления слов из n-грамма конкатенирутся. Есть также опцииональный residual connection мимо основного tanh преобразования ($Wx$):

$$P(wₜ | \text{контекст}) = \text{softmax}(b + W·x + U·tanh(d + H·x))\text{, где } x = [C(w_{t-3}), C(w_{t-2}), C(w_{t-1})]$$

Здесь `C` — матрица эмбеддингов размера `|V|×m`, общая для всех входных позиций; `H` и `U` — веса скрытого и выходного слоёв; 

Модель авторегрессионная и генеративная. На каждом шаге генерации на вход подается сгенерированные к этому моменту окно текста. На выходе к вектору размера |V| применяется softmax для получения вероятностей . На обучении оптимизируется лог-правдоподобие наблюдаемых данных. Позиция слова никак не учитывается.

На корпусах Brown и AP News нейросестевая модель по перплексии стала лучше n-граммных больше чем на 10%. Но главное была доказана идея: обучаемые распределённые представления работают и побеждают подсчёт частот

Ограничения модели:
- softmax по всему словарю на каждом шаге — самая дорогая операция: при словаре в десятки тысяч слов это доминирует в вычислениях<br>Бенжио боролся с этим распараллеливанием, а последующие работы придумали способы обойти полный softmax: иерархический softmax (Морен и Бенжио, 2005), важностную выборку, а позже noise-contrastive estimation и negative sampling — последнее и сделало возможным быстрый word2vec<br><Br>
- главное ограничение языковой модели: фиксированный и небольшой контекст<br>позже это ограничение обошли с помощью рекурсивных сетей RNN, LSTM

Из идеи обучения распределенного представления позднее выросли word2vec (Миколов, 2013), GloVe а также обучаемые эмбеддинги во всех современных трансформерех